<a href="https://colab.research.google.com/github/lsmc-isa/avcad_2026/blob/main/Exercise_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile
import os
import requests

In [2]:
from google.colab import files
import zipfile
import os
import pandas as pd

output_dir = 'EFIplus_medit_data'

# Create directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

print("Para extrair os dados do seu computador local, você precisa fazer o upload do arquivo para o Colab.")
print("Por favor, clique em 'Choose Files' para fazer o upload de 'EFIplus_medit.zip' da sua pasta de Downloads.")

# This will open a file picker in the user's browser
uploaded = files.upload()

# Check if file was uploaded
if uploaded:
    # Get the name of the first uploaded file (assuming only one file is uploaded at a time)
    uploaded_file_name = list(uploaded.keys())[0]
    zip_file_path = f'/content/{uploaded_file_name}' # The uploaded file will be in /content/

    print(f"Upload de '{uploaded_file_name}' realizado com sucesso.")
    print(f"Tentando descompactar '{zip_file_path}'...")
    try:
        with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
            zip_ref.extractall(output_dir)
        print("Extração completa!")
    except zipfile.BadZipFile:
        print(f"Erro: O arquivo enviado '{uploaded_file_name}' não é um arquivo ZIP válido. Por favor, certifique-se de fazer o upload de um arquivo ZIP não corrompido.")
    except Exception as e:
        print(f"Ocorreu um erro inesperado durante a extração: {e}")
else:
    print("Nenhum arquivo foi enviado. Por favor, faça o upload do arquivo 'EFIplus_medit.zip' para continuar.")

Para extrair os dados do seu computador local, você precisa fazer o upload do arquivo para o Colab.
Por favor, clique em 'Choose Files' para fazer o upload de 'EFIplus_medit.zip' da sua pasta de Downloads.


Saving EFIplus_medit (1).zip to EFIplus_medit (1).zip
Upload de 'EFIplus_medit (1).zip' realizado com sucesso.
Tentando descompactar '/content/EFIplus_medit (1).zip'...
Extração completa!


## Análise de Dados EFIplus_medit

O arquivo `EFIplus_medit.zip` foi enviado e descompactado com sucesso. Os dados estão agora disponíveis no arquivo `EFIplus_medit.csv` localizado em `/content/EFIplus_medit_data/EFIplus_medit.csv`. Estamos prontos para carregar este arquivo para um DataFrame pandas e iniciar a análise.

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.preprocessing import StandardScaler
import seaborn as sns

# Load the CSV file into a pandas DataFrame, explicitly specifying semicolon as delimiter
# Using engine='python' for robustness with separator detection
df = pd.read_csv('/content/EFIplus_medit_data/EFIplus_medit.csv', sep=';', engine='python')

print("Primeiras 5 linhas do dataset:")
display(df.head())

print("\nInformações sobre o dataset (nomes das colunas, tipos de dados, valores não nulos):")
df.info()

# --- Identificação e Filtragem de Dados --- #
# It's crucial to correctly identify the basin column and quantitative environmental variables.
# We assume there's a 'Catchment_name' column (or similar) and that environmental variables are numeric.

# Identify the basin column. 'Catchment_name' is a strong candidate based on problem description.
basin_column_candidates = ['Catchment_name', 'Basin', 'River_Basin', 'Location_Group']
found_basin_column = None
for col_candidate in basin_column_candidates:
    if col_candidate in df.columns:
        found_basin_column = col_candidate
        break

if found_basin_column:
    basin_column = found_basin_column
    print(f"Coluna da bacia identificada como: '{basin_column}'.")
else:
    print(f"Erro: Nenhuma coluna de bacia conhecida foi encontrada. Colunas disponíveis: {df.columns.tolist()}")
    print("Por favor, edite manualmente a célula de código para definir 'basin_column' corretamente.")
    # Fallback to prevent further errors, but this will likely make subsequent steps fail
    # if no proper basin column is found.
    basin_column = None
    data_for_clustering = pd.DataFrame() # No valid data for clustering if basin column isn't found.

if basin_column:
    # Filter sites from Douro and Tejo basins
    # Assuming values in the basin column are 'Douro' and 'Tejo'
    sites_douro_tejo = df[df[basin_column].isin(['Douro', 'Tejo'])].copy()

    if sites_douro_tejo.empty:
        print(f"\nAtenção: Não foram encontrados sites nas bacias 'Douro' ou 'Tejo' usando a coluna '{basin_column}'.")
        print("Verifique os valores únicos na coluna da bacia:", df[basin_column].unique())
        print("Pode ser necessário ajustar os nomes das bacias ou a coluna 'basin_column' ou os nomes das bacias no filtro.")
        # If no specific basins found, attempt to use all numeric data for clustering but issue a warning.
        print("Prosseguindo com clustering usando todos os dados numéricos disponíveis (sem filtragem específica de bacia).")
        data_for_clustering = df.select_dtypes(include=np.number).copy()
        # Ensure 'Site_code' is not treated as a variable if it's numeric
        if 'Site_code' in data_for_clustering.columns:
            data_for_clustering = data_for_clustering.drop(columns=['Site_code'])
        site_labels = df['Site_code'].fillna('Unknown') if 'Site_code' in df.columns else df.index.astype(str)
        # Handle cases where `site_labels` might have duplicates if it's not unique identifier initially
        site_labels = site_labels.astype(str) + '_' + df.index.astype(str) if site_labels.duplicated().any() else site_labels
    else:
        print(f"\nSites das bacias do Douro e Tejo selecionados: {len(sites_douro_tejo)} registros.")
        # Identify quantitative environmental variables
        # Exclude basin column and any non-numeric ID columns
        id_cols_to_exclude = [col for col in sites_douro_tejo.columns if 'site_code' in col.lower() or 'country' in col.lower()]

        # Select only numeric columns that are not the basin column or inferred ID columns
        # Ensure we drop only existing columns
        cols_to_drop = [basin_column] + [col for col in id_cols_to_exclude if col in sites_douro_tejo.columns]
        quantitative_variables = sites_douro_tejo.select_dtypes(include=np.number).drop(columns=cols_to_drop, errors='ignore')

        # Remove columns with NaN values to ensure clustering works
        quantitative_variables = quantitative_variables.dropna(axis=1)

        if quantitative_variables.empty:
            print("Erro: Não foram encontradas variáveis ambientais quantitativas após filtragem e remoção de NaNs.")
            print("Verifique se as colunas numéricas representam variáveis ambientais e ajuste a lógica de filtragem.")
            display(sites_douro_tejo.select_dtypes(include=np.number).head())
            data_for_clustering = pd.DataFrame() # Set empty to prevent errors
        else:
            print(f"\nVariáveis ambientais quantitativas selecionadas para clustering: {quantitative_variables.columns.tolist()}")
            data_for_clustering = quantitative_variables
            site_labels = sites_douro_tejo['Site_code'].fillna('Unknown') if 'Site_code' in sites_douro_tejo.columns else sites_douro_tejo.index.astype(str)
            # Handle potential duplicate site labels
            site_labels = site_labels.astype(str) + '_' + sites_douro_tejo.index.astype(str) if site_labels.duplicated().any() else site_labels
else: # If basin_column is None
    data_for_clustering = pd.DataFrame()


# Scale the data so that all variables contribute equally to the distance calculation
if not data_for_clustering.empty:
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(data_for_clustering)
    # Ensure site_labels has the same length as scaled_data
    if len(site_labels) == scaled_data.shape[0]:
        scaled_df = pd.DataFrame(scaled_data, columns=data_for_clustering.columns, index=site_labels.values)
    else:
        print("Atenção: O número de labels dos sites não corresponde ao número de linhas dos dados escalados. Usando índices numéricos.")
        scaled_df = pd.DataFrame(scaled_data, columns=data_for_clustering.columns)
    print("\nDados escalados para análise de cluster.")
    display(scaled_df.head())
else:
    print("Não há dados válidos para clustering após a preparação. 'scaled_df' está vazio.")
    scaled_df = pd.DataFrame() # Ensure scaled_df is defined as empty if no data

Primeiras 5 linhas do dataset:


,Site_code,Latitude,Longitude,Country,Catchment_name,Galiza,Subsample,Calib_EFI_Medit,Calib_connect,Calib_hydrol,...,Squalius malacitanus,Squalius pyrenaicus,Squalius torgalensis,Thymallus thymallus,Tinca tinca,Zingel asper,Squalius sp,Barbatula sp,Phoxinus sp,Iberochondrostoma_sp
0,ES_01_0002,38.102003,-4.096070,Spain,Guadalquivir,0,1,0,1,0,...,0,0,0,0,0,0,0,0,0,0
1,ES_02_0001,40.530188,-1.887796,Spain,Tejo,0,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
2,ES_02_0002,40.595432,-1.928079,Spain,Tejo,0,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
3,ES_02_0003,40.656184,-1.989831,Spain,Tejo,0,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
4,ES_02_0004,40.676402,-2.036274,Spain,Tejo,0,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0



Informações sobre o dataset (nomes das colunas, tipos de dados, valores não nulos):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5011 entries, 0 to 5010
Columns: 164 entries, Site_code to Iberochondrostoma_sp
dtypes: float64(38), int64(120), object(6)
memory usage: 6.3+ MB
Coluna da bacia identificada como: 'Catchment_name'.

Sites das bacias do Douro e Tejo selecionados: 910 registros.

Variáveis ambientais quantitativas selecionadas para clustering: ['Latitude', 'Longitude', 'Galiza', 'Subsample', 'Calib_EFI_Medit', 'Calib_connect', 'Calib_hydrol', 'Calib_morphol', 'Calib_wqual', 'Altitude', 'Barriers_catchment_down', 'Barriers_river_segment_up', 'Barriers_river_segment_down', 'Barriers_number_river_segment_up', 'Barriers_number_river_segment_down', 'Barriers_distance_river_segment_up', 'Barriers_distance_river_segment_down', 'Impoundment', 'Hydropeaking', 'Hydro_mod', 'Temperature_impact', 'Velocity_increase', 'Reservoir_flushing', 'Channelisation', 'Riparian_vegetation', 'Emb

,Latitude,Longitude,Galiza,Subsample,Calib_EFI_Medit,Calib_connect,Calib_hydrol,Calib_morphol,Calib_wqual,Altitude,...,Squalius malacitanus,Squalius pyrenaicus,Squalius torgalensis,Thymallus thymallus,Tinca tinca,Zingel asper,Squalius sp,Barbatula sp,Phoxinus sp,Iberochondrostoma_sp
ES_02_0001,-0.096271,2.181967,-0.19398,0.171499,2.224345,0.4246,1.694798,1.170381,1.131681,1.935372,...,0.0,-0.617155,0.0,0.0,-0.110616,0.0,-1.084912,-0.033168,-0.110616,-0.275046
ES_02_0002,-0.027030,2.161110,-0.19398,0.171499,2.224345,0.4246,1.694798,1.170381,1.131681,1.665058,...,0.0,-0.617155,0.0,0.0,-0.110616,0.0,-1.084912,-0.033168,-0.110616,-0.275046
ES_02_0003,0.037444,2.129136,-0.19398,0.171499,2.224345,0.4246,1.694798,1.170381,1.131681,1.441636,...,0.0,-0.617155,0.0,0.0,-0.110616,0.0,-1.084912,-0.033168,-0.110616,-0.275046
ES_02_0004,0.058900,2.105089,-0.19398,0.171499,2.224345,0.4246,1.694798,1.170381,1.131681,1.287171,...,0.0,-0.617155,0.0,0.0,-0.110616,0.0,-1.084912,-0.033168,-0.110616,-0.275046
ES_02_0005,0.118785,2.083482,-0.19398,0.171499,2.224345,0.4246,1.694798,1.170381,1.131681,1.179598,...,0.0,-0.617155,0.0,0.0,-0.110616,0.0,-1.084912,-0.033168,-0.110616,-0.275046


## 1. Análise de Cluster Aglomerativa para Sites

Vamos realizar a análise de cluster aglomerativa nos sites filtrados (Douro e Tejo) usando diferentes métodos de ligação (`linkage`). Os métodos de ligação determinam como a distância entre os clusters é calculada. Comparar os resultados de diferentes métodos pode fornecer insights sobre a estrutura dos dados.

In [4]:
# Certifique-se de que scaled_df foi criado corretamente na célula anterior
if 'scaled_df' in locals() and not scaled_df.empty:
    print("Executando clustering com diferentes métodos de ligação:")

    linkage_methods = ['ward', 'complete', 'average', 'single']
    plt.figure(figsize=(15, 10))

    for i, method in enumerate(linkage_methods):
        plt.subplot(2, 2, i + 1)
        Z = linkage(scaled_df, method=method)
        dendrogram(Z, labels=scaled_df.index, leaf_rotation=90, leaf_font_size=8)
        plt.title(f'Dendrograma (Linkage: {method})')
        plt.ylabel('Distância')

    plt.tight_layout()
    plt.show()
else:
    print("Não há dados escalados disponíveis para realizar a análise de cluster aglomerativa. Certifique-se de que a célula anterior foi executada com sucesso e gerou 'scaled_df'.")

Não há dados escalados disponíveis para realizar a análise de cluster aglomerativa. Certifique-se de que a célula anterior foi executada com sucesso e gerou 'scaled_df'.


## 2. Heatmap e Dendrograma para Sites (Linkage Médio)

Agora, vamos gerar um heatmap combinado com um dendrograma para visualizar os clusters de sites (linhas) usando o método de ligação médio (`average linkage`). O heatmap mostra os valores das variáveis ambientais para cada site, enquanto o dendrograma agrupa os sites com base em sua similaridade.

In [5]:
if 'scaled_df' in locals() and not scaled_df.empty:
    print("Gerando Heatmap e Dendrograma para Sites (Linkage Médio):")
    sns.clustermap(scaled_df, method='average', cmap='viridis', figsize=(12, 10), row_cluster=True, col_cluster=False, dendrogram_ratio=(0.2, 0.0))
    plt.suptitle('Heatmap e Dendrograma de Sites (Linkage Médio)', y=1.02) # Ajusta o título para não sobrepor
    plt.show()
else:
    print("Não há dados escalados disponíveis para gerar o heatmap e dendrograma de sites. Certifique-se de que a célula anterior foi executada com sucesso e gerou 'scaled_df'.")

Não há dados escalados disponíveis para gerar o heatmap e dendrograma de sites. Certifique-se de que a célula anterior foi executada com sucesso e gerou 'scaled_df'.


## 3. Dendrograma para Variáveis Ambientais (Linkage Médio)

Em seguida, vamos gerar um dendrograma que agrupa as *variáveis ambientais* (colunas) em vez dos sites (linhas). Isso nos permite identificar quais variáveis ambientais são mais semelhantes entre si, o que pode ser útil para selecionar variáveis para modelos de regressão ou para reduzir a dimensionalidade dos dados.

In [6]:
if 'scaled_df' in locals() and not scaled_df.empty:
    print("Gerando Dendrograma para Variáveis Ambientais (Linkage Médio):")
    # Para agrupar as variáveis (colunas), precisamos transpor o DataFrame
    scaled_df_transposed = scaled_df.T

    # Realizar o linkage nas variáveis transpostas
    Z_vars = linkage(scaled_df_transposed, method='average')

    plt.figure(figsize=(10, 7))
    dendrogram(Z_vars, labels=scaled_df_transposed.index, leaf_rotation=90, leaf_font_size=10)
    plt.title('Dendrograma de Variáveis Ambientais (Linkage Médio)')
    plt.ylabel('Distância')
    plt.xlabel('Variáveis Ambientais')
    plt.tight_layout()
    plt.show()
else:
    print("Não há dados escalados disponíveis para gerar o dendrograma de variáveis ambientais. Certifique-se de que a célula anterior foi executada com sucesso e gerou 'scaled_df'.")

Não há dados escalados disponíveis para gerar o dendrograma de variáveis ambientais. Certifique-se de que a célula anterior foi executada com sucesso e gerou 'scaled_df'.


## Discussão sobre a Seleção de Variáveis com o Dendrograma

O dendrograma de variáveis ambientais (colunas) é uma ferramenta valiosa para a seleção de variáveis, especialmente em análises como regressão. Ele nos ajuda a:

*   **Identificar Redundância:** Variáveis que se agrupam em clusters muito próximos no dendrograma são altamente correlacionadas. Isso significa que elas fornecem informações semelhantes. Em um modelo de regressão, incluir variáveis altamente correlacionadas (multicolinearidade) pode levar a estimativas de coeficientes instáveis e dificuldade em interpretar a contribuição individual de cada variável.

*   **Redução de Dimensionalidade:** Ao invés de usar todas as variáveis de um cluster altamente correlacionado, pode-se escolher uma variável representativa do cluster, ou criar uma nova variável combinando-as (e.g., média, componente principal), o que simplifica o modelo sem perder muita informação.

*   **Seleção de Subconjuntos de Variáveis:** Podemos usar os cortes do dendrograma para identificar grupos de variáveis. Para cada grupo, podemos selecionar a variável mais interpretabilística ou com melhor desempenho em testes univariados, ou simplesmente escolher uma para representar o conjunto.

*   **Compreensão Estrutural:** O dendrograma revela a estrutura intrínseca de semelhança entre as variáveis. Por exemplo, variáveis relacionadas à qualidade da água podem formar um cluster, enquanto variáveis geomorfológicas formam outro. Isso pode guiar a construção de modelos mais teoricamente fundamentados.

Em resumo, ao observar o dendrograma das variáveis, podemos fazer escolhas informadas sobre quais variáveis incluir em modelos de regressão, evitando a redundância, simplificando o modelo e potencialmente melhorando sua estabilidade e interpretabilidade.